In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("demo").getOrCreate()

# Pivot

In [5]:
from pyspark.sql.functions import sum, first

In [2]:
data = [
    (101, "login", 1),
    (101, "purchase", 2),
    (102, "login", 1),
    (103, "logout", 1)
]

df = spark.createDataFrame(data, ["user_id", "event", "cnt"])
df.show()

+-------+--------+---+
|user_id|   event|cnt|
+-------+--------+---+
|    101|   login|  1|
|    101|purchase|  2|
|    102|   login|  1|
|    103|  logout|  1|
+-------+--------+---+



In [8]:
events = ["login", "logout", "purchase"]

df_events = (
    df.groupBy("user_id")
      .pivot("event", events)
      .agg(sum("cnt"))
      .fillna(0)
)

df_events.show()

+-------+-----+------+--------+
|user_id|login|logout|purchase|
+-------+-----+------+--------+
|    103|    0|     1|       0|
|    101|    1|     0|       2|
|    102|    1|     0|       0|
+-------+-----+------+--------+



## Example with duplicate matrics

In [16]:
data = [
    (1, "clicks", 10),
    (1, "views", 20),
    (1, "purchases", 30),
    (2, "clicks", 5),
    (2, "views", 15),
    (2, "purchases", 25),
    (2, "purchases", 5),
    (3, "purchases", 35)
]

df = spark.createDataFrame(data, ["id", "metric", "value"])
df.show()

+---+---------+-----+
| id|   metric|value|
+---+---------+-----+
|  1|   clicks|   10|
|  1|    views|   20|
|  1|purchases|   30|
|  2|   clicks|    5|
|  2|    views|   15|
|  2|purchases|   25|
|  2|purchases|    5|
|  3|purchases|   35|
+---+---------+-----+



In [17]:
df_pivot = (
    df.groupBy("id")
      .pivot("metric")
      .agg(sum("value"))
)

df_pivot.show()

+---+------+---------+-----+
| id|clicks|purchases|views|
+---+------+---------+-----+
|  1|    10|       30|   20|
|  3|  NULL|       35| NULL|
|  2|     5|       30|   15|
+---+------+---------+-----+



## Pivot with Known Column List
It provides better performance because it avoids a full distinct scan

In [18]:
metrics = ["clicks", "views", "purchases"]

df_pivot = (
    df.groupBy("id")
      .pivot("metric", metrics)
      .agg(sum("value"))
      .fillna(0)
)
df_pivot.show()

+---+------+-----+---------+
| id|clicks|views|purchases|
+---+------+-----+---------+
|  1|    10|   20|       30|
|  3|     0|    0|       35|
|  2|     5|   15|       30|
+---+------+-----+---------+



In [22]:
df_pivot = (
    df.groupBy("id")
      .pivot("metric", metrics)
      .agg(first("value"))
      .fillna(0)
)
df_pivot.show()

+---+------+-----+---------+
| id|clicks|views|purchases|
+---+------+-----+---------+
|  1|    10|   20|       30|
|  3|     0|    0|       35|
|  2|     5|   15|       25|
+---+------+-----+---------+



# Unpivot and array to rows
1. single column to rows
2. multiple columns to rows(Unpivot)

In [29]:

data = [
    (1, "Surya", "Python,SQL,Spark"),
    (2, "Surya1", "Java,SQL"),
    (3, "Surya2", "Python")
]

df = spark.createDataFrame(data, ["id", "name", "skills"])
df.show()

+---+------+----------------+
| id|  name|          skills|
+---+------+----------------+
|  1| Surya|Python,SQL,Spark|
|  2|Surya1|        Java,SQL|
|  3|Surya2|          Python|
+---+------+----------------+



In [30]:
from pyspark.sql.functions import explode, split, col

In [31]:
df.withColumn("skill", explode(split(col("skills"), ","))).show()

+---+------+----------------+------+
| id|  name|          skills| skill|
+---+------+----------------+------+
|  1| Surya|Python,SQL,Spark|Python|
|  1| Surya|Python,SQL,Spark|   SQL|
|  1| Surya|Python,SQL,Spark| Spark|
|  2|Surya1|        Java,SQL|  Java|
|  2|Surya1|        Java,SQL|   SQL|
|  3|Surya2|          Python|Python|
+---+------+----------------+------+



In [35]:
data = [
    (1, 10, 20, 30),
    (2, 5, 15, 25)
]

df = spark.createDataFrame(
    data, ["id", "clicks", "views", "purchases"]
)
df.show()


+---+------+-----+---------+
| id|clicks|views|purchases|
+---+------+-----+---------+
|  1|    10|   20|       30|
|  2|     5|   15|       25|
+---+------+-----+---------+



In [36]:
from pyspark.sql.functions import expr

df_unpivot = df.select(
    "id",
    expr("""
        stack(3,
            'clicks', clicks,
            'views', views,
            'purchases', purchases
        ) as (metric, value)
    """)
)

df_unpivot.show()


+---+---------+-----+
| id|   metric|value|
+---+---------+-----+
|  1|   clicks|   10|
|  1|    views|   20|
|  1|purchases|   30|
|  2|   clicks|    5|
|  2|    views|   15|
|  2|purchases|   25|
+---+---------+-----+



In [47]:
from pyspark.sql.functions import create_map, explode, lit

df_ = df.withColumn(
    "metrics_map",
    create_map(
        lit("clicks"), "clicks",
        lit("views"), df["views"],
        lit("purchases"), col("purchases")
    )
)

df_.show(truncate=False)

+---+------+-----+---------+--------------------------------------------+
|id |clicks|views|purchases|metrics_map                                 |
+---+------+-----+---------+--------------------------------------------+
|1  |10    |20   |30       |{clicks -> 10, views -> 20, purchases -> 30}|
|2  |5     |15   |25       |{clicks -> 5, views -> 15, purchases -> 25} |
+---+------+-----+---------+--------------------------------------------+



In [46]:

df_map = df_.select(
    "id",
    explode("metrics_map").alias("metric", "value")
)

df_map.show()


+---+---------+-----+
| id|   metric|value|
+---+---------+-----+
|  1|   clicks|   10|
|  1|    views|   20|
|  1|purchases|   30|
|  2|   clicks|    5|
|  2|    views|   15|
|  2|purchases|   25|
+---+---------+-----+



## unpivot with SQL

In [48]:
df.createOrReplaceTempView("metrics")

df_unpivot = spark.sql("""
    SELECT
        id,
        metric,
        value
    FROM metrics
    LATERAL VIEW
        stack(
            3,
            'clicks', clicks,
            'views', views,
            'purchases', purchases
        ) AS metric, value
""")

df_unpivot.show()

+---+---------+-----+
| id|   metric|value|
+---+---------+-----+
|  1|   clicks|   10|
|  1|    views|   20|
|  1|purchases|   30|
|  2|   clicks|    5|
|  2|    views|   15|
|  2|purchases|   25|
+---+---------+-----+



# Union

In [52]:

data_1 = [
    (1, "surya0", 100),
    (2, "surya1", 200)
]

data_2 = [
    (3, "surya2", 150),
    (2, "surya1", 200)
]

df_1 = spark.createDataFrame(data_1, ["id", "name", "sales"])
df_2 = spark.createDataFrame(data_2, ["id", "name", "sales"])

df_1.show()
df_2.show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  1|surya0|  100|
|  2|surya1|  200|
+---+------+-----+

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  3|surya2|  150|
|  2|surya1|  200|
+---+------+-----+



In [53]:
df_all = df_1.union(df_2)
df_all.show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  1|surya0|  100|
|  2|surya1|  200|
|  3|surya2|  150|
|  2|surya1|  200|
+---+------+-----+



Union gives all values with duplicates

In [56]:
#### use distinct to avoid duplicates
df_all.distinct().show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  1|surya0|  100|
|  2|surya1|  200|
|  3|surya2|  150|
+---+------+-----+



Column should be in same order

In [58]:
df_a = spark.createDataFrame(
    [(1, "surya1", 100)],
    ["id", "name", "sales"]
)

df_b = spark.createDataFrame(
    [(200, "surya2", 2)],
    ["sales", "name", "id"]
)

df_a.show()
df_b.show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  1|surya1|  100|
+---+------+-----+

+-----+------+---+
|sales|  name| id|
+-----+------+---+
|  200|surya2|  2|
+-----+------+---+



In [61]:
df_a.union(df_b).show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  1|surya1|  100|
|200|surya2|    2|
+---+------+-----+



df_a.union(df_b).show() worked in the above case because the physical column order in both DataFrames is actually the same at execution time.
It might have worked on the above case, but union() in Spark is positional, not name-based. Even if schemas look compatible, they can silently produce incorrect data when column orders differ. That’s why in production I always prefer unionByName().
Let's add few more columns with incompatable types.


In [76]:
df_a = spark.createDataFrame(
    [(1, "surya1", 100, "abc", "d", 5)],
    ["id", "name", "sales", "col4", "col5", "col6" ]
)

df_b = spark.createDataFrame(
    [(200, "surya2", 2, "a", 6, "b")],
    ["sales", "name", "id", "col5", "col6", "col4"]
)

df_a.show()
df_b.show()

+---+------+-----+----+----+----+
| id|  name|sales|col4|col5|col6|
+---+------+-----+----+----+----+
|  1|surya1|  100| abc|   d|   5|
+---+------+-----+----+----+----+

+-----+------+---+----+----+----+
|sales|  name| id|col5|col6|col4|
+-----+------+---+----+----+----+
|  200|surya2|  2|   a|   6|   b|
+-----+------+---+----+----+----+



In [84]:
# this time it is highly likely that it will fail. The schema for both dataframes is the same, just the columns are not in the same order.
try:
    # code that may fail
    df_a.union(df_b).show()
except Exception as e:
    print(str(e))

{"ts": "2026-01-17 23:23:33.658", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'd' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o1105.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'd' of the type \"STRING\" cannot be cast to \"BIGINT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:147)\n\tat org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withExcept

[CAST_INVALID_INPUT] The value 'd' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018


In [86]:
# unionByName for the same dataset will work
df_union = df_a.unionByName(df_b)
df_union.show()

+---+------+-----+----+----+----+
| id|  name|sales|col4|col5|col6|
+---+------+-----+----+----+----+
|  1|surya1|  100| abc|   d|   5|
|  2|surya2|  200|   b|   a|   6|
+---+------+-----+----+----+----+



In [89]:
df_old = spark.createDataFrame(
    [(1, "Surya1", 100)],
    ["id", "name", "sales"]
)

df_new = spark.createDataFrame(
    [(2, "surya2", 200, "US")],
    ["id", "name", "sales", "country"]
)

In [90]:
df_union = df_old.unionByName(df_new, allowMissingColumns=True)
df_union.show()

+---+------+-----+-------+
| id|  name|sales|country|
+---+------+-----+-------+
|  1|Surya1|  100|   NULL|
|  2|surya2|  200|     US|
+---+------+-----+-------+



## Union many dataframes


In [92]:
list_of_dfs = []

for i in range(10):
    df_ = spark.createDataFrame(
        [(i, f"surya{i}", 100+i)],
        ["id", "name", "sales"]
    )
    list_of_dfs.append(df_)


In [93]:
from functools import reduce

df_final = reduce(lambda a, b: a.unionByName(b), list_of_dfs)
df_final.show()

+---+------+-----+
| id|  name|sales|
+---+------+-----+
|  0|surya0|  100|
|  1|surya1|  101|
|  2|surya2|  102|
|  3|surya3|  103|
|  4|surya4|  104|
|  5|surya5|  105|
|  6|surya6|  106|
|  7|surya7|  107|
|  8|surya8|  108|
|  9|surya9|  109|
+---+------+-----+

